In [2]:



from scripts.storage.config.paths import get_path_builder, SourceSystem
from scripts.storage.config.spark import get_spark_session

spark = get_spark_session(app_name="postgres_silver_eda")


path_builder = get_path_builder()
import os
ingestion_date = os.getenv("INGESTION_DATE", "2026-11-15")


print(f"Spark Session: {spark.sparkContext.appName}")
print(f"Ingestion Date: {ingestion_date}")

Spark Session: postgres_silver_eda
Ingestion Date: 2026-11-15


In [3]:
# Cell 2: Reading Postgres tables and running the stable unified comprehensive profiling
# -----------------------------------------------------------------
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import NumericType, StringType, BooleanType, DateType, TimestampType
from typing import List, Dict

# 1. Building paths and reading data based on dynamic date (singular for subscription)
tables = ["users", "subscription", "payments"]
dfs: Dict[str, DataFrame] = {}

print("Reading data from Bronze Layer...")
for table in tables:
    path = path_builder.bronze_partitioned(
        source=SourceSystem.POSTGRES,
        ingestion_date=ingestion_date,
        table=table
    )
    dfs[table] = spark.read.parquet(path)
    print(f"Loaded table {table}: ({dfs[table].count():,} rows)")

df_users = dfs["users"]
df_subscriptions = dfs["subscription"]
df_payments = dfs["payments"]

# 2. Stable unified single-pass comprehensive profiling class (ProfileTable)
class ProfileTable:
    def __init__(self, df: DataFrame, table_name: str, primary_key: str):
        self.df = df
        self.table_name = table_name
        self.primary_key = primary_key
        self.schema = df.schema
        self._build_expressions()

    def _build_expressions(self) -> None:
        """Building profiling expressions with type unification to double to avoid merge errors in Spark"""
        self.exprs: List[F.Column] = []
        self.numeric_cols: List[str] = []
        self.categorical_cols: List[str] = []

        for field in self.schema.fields:
            col_name = field.name
            dtype = field.dataType

            # Basic checks for all columns
            self.exprs.extend([
                F.count(F.col(col_name)).alias(f"{col_name}__count"),
                F.sum(F.when(F.col(col_name).isNull(), 1).otherwise(0)).alias(f"{col_name}__null_count"),
                F.countDistinct(F.col(col_name)).alias(f"{col_name}__distinct_count"),
            ])

            # Advanced statistics for numeric columns (with cast to double to definitively solve merge issue)
            if isinstance(dtype, NumericType):
                self.numeric_cols.append(col_name)
                col_expr = F.col(col_name).cast("double")
                self.exprs.extend([
                    F.min(col_expr).alias(f"{col_name}__min"),
                    F.max(col_expr).alias(f"{col_name}__max"),
                    F.mean(col_expr).alias(f"{col_name}__mean"),
                    F.stddev(col_expr).alias(f"{col_name}__stddev"),
                    F.expr(f"percentile_approx(cast({col_name} as double), array(0.25, 0.5, 0.75))").alias(f"{col_name}__quantiles"),
                ])
            elif isinstance(dtype, (StringType, DateType, TimestampType)):
                self.categorical_cols.append(col_name)
            elif isinstance(dtype, BooleanType):
                self.exprs.extend([
                    F.sum(F.when(F.col(col_name) == True, 1).otherwise(0)).alias(f"{col_name}__true_count"),
                    F.sum(F.when(F.col(col_name) == False, 1).otherwise(0)).alias(f"{col_name}__false_count"),
                ])

        # Calculating full row duplication
        self.exprs.append(
            (F.count("*") - F.countDistinct(F.concat_ws("||", *self.df.columns))).alias("__exact_duplicates")
        )

    def run(self) -> None:
        """Running profiling and printing comprehensive report"""
        print(f"\n{'='*80}\nComprehensive profiling report for table: {self.table_name.upper()}\n{'='*80}")
        self.df.printSchema()

        # Executing calculations in a single pass
        agg_row = self.df.agg(*self.exprs).collect()[0]
        total_rows = agg_row[f"{self.schema.fields[0].name}__count"]

        # Null and cardinality report
        print(f"\nNULL and CARDINALITY report for table - {self.table_name}")
        null_rows = []
        for field in self.schema.fields:
            col_name = field.name
            cnt = agg_row[f"{col_name}__count"]
            nulls = agg_row[f"{col_name}__null_count"]
            distinct = agg_row[f"{col_name}__distinct_count"]
            null_rows.append({
                "Column": col_name,
                "Type": str(field.dataType).replace("Type()", ""),
                "Count": cnt,
                "Null_Count": nulls,
                "Null_Pct %": round(nulls / total_rows * 100, 2) if total_rows else 0.0,
                "Distinct_Count": distinct,
            })
        spark.createDataFrame(null_rows).show(truncate=False)

        # Stable advanced numeric statistics report
        if self.numeric_cols:
            print(f"\nNumeric analytical statistics - {self.table_name}")
            num_rows = []
            for col_name in self.numeric_cols:
                q = agg_row[f"{col_name}__quantiles"]
                
                # Safe conversion of Python values to float
                q25 = float(q[0]) if (q and q[0] is not None) else None
                q50 = float(q[1]) if (q and q[1] is not None) else None
                q75 = float(q[2]) if (q and q[2] is not None) else None

                num_rows.append({
                    "Column": col_name,
                    "Min": float(agg_row[f"{col_name}__min"]) if agg_row[f"{col_name}__min"] is not None else None,
                    "Max": float(agg_row[f"{col_name}__max"]) if agg_row[f"{col_name}__max"] is not None else None,
                    "Mean": round(float(agg_row[f"{col_name}__mean"]), 2) if agg_row[f"{col_name}__mean"] else None,
                    "StdDev": round(float(agg_row[f"{col_name}__stddev"]), 2) if agg_row[f"{col_name}__stddev"] else None,
                    "Q25%": q25,
                    "Q50% (Median)": q50,
                    "Q75%": q75,
                })
            spark.createDataFrame(num_rows).show(truncate=False)

        # Top 5 most frequent values for text columns
        if self.categorical_cols:
            print(f"\nTop 5 most frequent values (Top-5 Frequencies) - {self.table_name}")
            for col_name in self.categorical_cols:
                print(f"  Column: {col_name}")
                self.df.groupBy(col_name).count().orderBy(F.desc("count")).show(5, truncate=False)

        # Primary key integrity and horizontal duplicates check
        print(f"\nPrimary key duplication and integrity and rows check - {self.table_name}")
        pk_nulls = agg_row[f"{self.primary_key}__null_count"]
        pk_distinct = agg_row[f"{self.primary_key}__distinct_count"]
        exact_dupes = agg_row["__exact_duplicates"]
        pk_dupes = total_rows - pk_distinct

        summary_rows = [
            {"Metric": "Total Rows", "Value": total_rows},
            {"Metric": f"Null values in key ({self.primary_key} Nulls)", "Value": pk_nulls},
            {"Metric": f"Duplicate keys ({self.primary_key} Duplicates)", "Value": pk_dupes},
            {"Metric": "Fully duplicated rows (Exact Row Duplicates)", "Value": exact_dupes}
        ]
        spark.createDataFrame(summary_rows).show(truncate=False)

# 3. Running actual profiling on the three tables in order with matching names
pk_map = {"users": "user_id", "subscription": "subscription_id", "payments": "payment_id"}
for table_name, pk in pk_map.items():
    profiler = ProfileTable(df=dfs[table_name], table_name=table_name, primary_key=pk)
    profiler.run()

# 4. Referential integrity and relationships check between tables
print(f"\n{'='*80}\nMissing relationships check between tables (Referential Integrity)\n{'='*80}")
orphan_subs = df_subscriptions.join(df_users.select("user_id"), on="user_id", how="left_anti").count()
print(f"  Orphan subscription rows without a real user (Orphan Subscriptions): {orphan_subs}")
orphan_payments = df_payments.join(df_subscriptions.select("subscription_id"), on="subscription_id", how="left_anti").count()
print(f"  Orphan payment rows without a real subscription (Orphan Payments): {orphan_payments}")
print(f"\n{'='*80}\nPOSTGRESQL exploration completed successfully\n{'='*80}")

Reading data from Bronze Layer...


26/11/15 11:05:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Loaded table users: (10,000 rows)
Loaded table subscription: (11,504 rows)
Loaded table payments: (6,696 rows)

Comprehensive profiling report for table: USERS
root
 |-- user_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- country: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- created_at: timestamp (nullable = true)
 |-- last_login: timestamp (nullable = true)



26/11/15 11:05:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                


NULL and CARDINALITY report for table - users


+--------------+-----+--------------+----------+----------+---------+
|Column        |Count|Distinct_Count|Null_Count|Null_Pct %|Type     |
+--------------+-----+--------------+----------+----------+---------+
|user_id       |10000|10000         |0         |0.0       |String   |
|email         |9939 |9661          |61        |0.61      |String   |
|country       |10000|8             |0         |0.0       |String   |
|account_status|10000|4             |0         |0.0       |String   |
|created_at    |10000|10000         |0         |0.0       |Timestamp|
|last_login    |10000|9998          |0         |0.0       |Timestamp|
+--------------+-----+--------------+----------+----------+---------+


Top 5 most frequent values (Top-5 Frequencies) - users
  Column: user_id
+------------------------------------+-----+
|user_id                             |count|
+------------------------------------+-----+
|01a1dedc-292e-7857-83ae-9b9394b26675|1    |
|01a1dedc-2932-727a-a2be-9512f307be1c|1    |



NULL and CARDINALITY report for table - subscription
+---------------+-----+--------------+----------+----------+---------+
|Column         |Count|Distinct_Count|Null_Count|Null_Pct %|Type     |
+---------------+-----+--------------+----------+----------+---------+
|subscription_id|11504|11504         |0         |0.0       |String   |
|user_id        |11504|10000         |0         |0.0       |String   |
|plan_type      |11504|3             |0         |0.0       |String   |
|start_date     |11504|11504         |0         |0.0       |Timestamp|
|end_date       |11504|9662          |0         |0.0       |Timestamp|
|is_active      |11504|2             |0         |0.0       |Boolean  |
|auto_renew     |11504|2             |0         |0.0       |Boolean  |
+---------------+-----+--------------+----------+----------+---------+


Top 5 most frequent values (Top-5 Frequencies) - subscription
  Column: subscription_id
+------------------------------------+-----+
|subscription_id              


NULL and CARDINALITY report for table - payments
+---------------+-----+--------------+----------+----------+------------------+
|Column         |Count|Distinct_Count|Null_Count|Null_Pct %|Type              |
+---------------+-----+--------------+----------+----------+------------------+
|payment_id     |6696 |6696          |0         |0.0       |String            |
|subscription_id|6696 |5714          |0         |0.0       |String            |
|user_id        |6696 |4962          |0         |0.0       |String            |
|amount         |6696 |6             |0         |0.0       |DecimalType(38,18)|
|currency       |6696 |1             |0         |0.0       |String            |
|payment_date   |6696 |6696          |0         |0.0       |Timestamp         |
|payment_status |6696 |4             |0         |0.0       |String            |
|attempt_no     |6696 |10            |0         |0.0       |Integer           |
|billing_reason |6696 |4             |0         |0.0       |String    

In [4]:
from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import NumericType, StringType, BooleanType, DateType, TimestampType
from typing import List, Dict


tables = ["users", "subscription", "payments"]
dfs: Dict[str, DataFrame] = {}

print("Reading data from Bronze Layer...")
for table in tables:
    path = path_builder.bronze_partitioned(
        source=SourceSystem.POSTGRES,
        ingestion_date=ingestion_date,
        table=table
    )
    dfs[table] = spark.read.parquet(path)
    print(f"Loaded table {table}: ({dfs[table].count():,} rows)")

df_users = dfs["users"]
df_subscriptions = dfs["subscription"]
df_payments = dfs["payments"]
# =============================================================================
# SUPPLEMENTARY: DIRTY DATA INJECTION PROFILING — POSTGRESQL TABLES
# Targets anomalies from PostgresDirtyDataInjector across users, subscriptions, payments
# =============================================================================

print("=" * 80)
print("DIRTY DATA INJECTION PROFILING — POSTGRESQL TABLES")
print("=" * 80)

# ============================================================
# USERS TABLE ANOMALIES
# ============================================================
print("\n" + "=" * 80)
print("USERS TABLE — Dirty Data Injection Profiling")
print("=" * 80)

# Anomaly Type 1: Null mandatory field — email
null_email = df_users.filter(F.col("email").isNull()).count()
print(f"\n[User Anomaly 1] Null email: {null_email:,}")

# Anomaly Type 5: Empty email (whitespace only)
empty_email = df_users.filter(F.trim(F.col("email")) == "").count()
print(f"[User Anomaly 5] Empty/whitespace email: {empty_email:,}")

# Anomaly Type 2: Invalid country enum ("XXX")
invalid_country = df_users.filter(F.col("country") == "XXX").count()
print(f"\n[User Anomaly 2] Invalid country='XXX': {invalid_country:,}")

print("\nFull country distribution (including dirty):")
df_users.groupBy("country").count().orderBy(F.desc("count")).show(truncate=False)

# Anomaly Type 3: Temporal skew — last_login < created_at
login_before_created = df_users.filter(
    F.col("last_login").isNotNull() & 
    F.col("created_at").isNotNull() & 
    (F.col("last_login") < F.col("created_at"))
).count()
print(f"\n[User Anomaly 3] last_login before created_at: {login_before_created:,}")

if login_before_created > 0:
    df_users.filter(F.col("last_login") < F.col("created_at")) \
        .select("user_id", "created_at", "last_login") \
        .show(20, truncate=False)

# Anomaly Type 4: Invalid account_status enum ("INVALID_STATUS")
invalid_acc_status = df_users.filter(F.col("account_status") == "INVALID_STATUS").count()
print(f"\n[User Anomaly 4] Invalid account_status='INVALID_STATUS': {invalid_acc_status:,}")

print("\nFull account_status distribution (including dirty):")
df_users.groupBy("account_status").count().orderBy(F.desc("count")).show(truncate=False)

# ============================================================
# SUBSCRIPTIONS TABLE ANOMALIES
# ============================================================
print("\n" + "=" * 80)
print("SUBSCRIPTIONS TABLE — Dirty Data Injection Profiling")
print("=" * 80)

# Anomaly Type 1: Invalid plan_type enum ("INVALID_PLAN")
invalid_plan = df_subscriptions.filter(F.col("plan_type") == "INVALID_PLAN").count()
print(f"\n[Sub Anomaly 1] Invalid plan_type='INVALID_PLAN': {invalid_plan:,}")

print("\nFull plan_type distribution (including dirty):")
df_subscriptions.groupBy("plan_type").count().orderBy(F.desc("count")).show(truncate=False)

# Anomaly Type 2: Temporal skew — end_date < start_date
end_before_start = df_subscriptions.filter(
    F.col("end_date").isNotNull() & 
    F.col("start_date").isNotNull() & 
    (F.col("end_date") < F.col("start_date"))
).count()
print(f"\n[Sub Anomaly 2] end_date before start_date: {end_before_start:,}")

if end_before_start > 0:
    df_subscriptions.filter(F.col("end_date") < F.col("start_date")) \
        .select("subscription_id", "user_id", "start_date", "end_date") \
        .show(20, truncate=False)

# DISABLED Anomaly 3: Invalid is_active (non-boolean) - verify not present
# Check is_active is clean boolean
is_active_dist = df_subscriptions.groupBy("is_active").count().collect()
print(f"\n[Disabled Anomaly 3 Check] is_active distribution: {is_active_dist}")

# ============================================================
# PAYMENTS TABLE ANOMALIES
# ============================================================
print("\n" + "=" * 80)
print("PAYMENTS TABLE — Dirty Data Injection Profiling")
print("=" * 80)

# Anomaly Type 1: Invalid payment_status enum ("INVALID_STATUS")
invalid_pay_status = df_payments.filter(F.col("payment_status") == "INVALID_STATUS").count()
print(f"\n[Pay Anomaly 1] Invalid payment_status='INVALID_STATUS': {invalid_pay_status:,}")

# Anomaly Type 2: Invalid billing_reason enum ("INVALID_REASON")
invalid_bill_reason = df_payments.filter(F.col("billing_reason") == "INVALID_REASON").count()
print(f"[Pay Anomaly 2] Invalid billing_reason='INVALID_REASON': {invalid_bill_reason:,}")

# Anomaly Type 3: Valid but wrong payment_status (random valid value)
# Anomaly Type 4: Valid but wrong billing_reason (random valid value)
# These are harder to detect automatically - show full distributions
print("\nFull payment_status distribution (including dirty):")
df_payments.groupBy("payment_status").count().orderBy(F.desc("count")).show(truncate=False)

print("\nFull billing_reason distribution (including dirty):")
df_payments.groupBy("billing_reason").count().orderBy(F.desc("count")).show(truncate=False)

# Anomaly Type 5: Attempt_no out of bounds (should be 1-3, injector sets 4-10)
invalid_attempt = df_payments.filter(
    (F.col("attempt_no") < 1) | (F.col("attempt_no") > 3)
).count()
print(f"\n[Pay Anomaly 5] attempt_no out of bounds (not 1-3): {invalid_attempt:,}")

if invalid_attempt > 0:
    df_payments.filter((F.col("attempt_no") < 1) | (F.col("attempt_no") > 3)) \
        .select("payment_id", "attempt_no") \
        .show(20, truncate=False)

print("\nAttempt_no distribution:")
df_payments.groupBy("attempt_no").count().orderBy("attempt_no").show(truncate=False)

# Anomaly Type 6: Negative amount
neg_amount = df_payments.filter(F.col("amount") < 0).count()
print(f"\n[Pay Anomaly 6] Negative amount: {neg_amount:,}")

if neg_amount > 0:
    df_payments.filter(F.col("amount") < 0) \
        .select("payment_id", "amount", "payment_status") \
        .show(20, truncate=False)

# Amount statistics
amt_stats = df_payments.agg(
    F.min("amount").alias("min"),
    F.max("amount").alias("max"),
    F.mean("amount").alias("mean")
).collect()[0]
print(f"\nAmount stats: min={amt_stats['min']}, max={amt_stats['max']}, mean={amt_stats['mean']:.2f}")

# DISABLED Anomaly 7: Referential orphan — subscription_id/user_id mutation
# Verify no empty strings
empty_sub_id = df_payments.filter(F.trim(F.col("subscription_id")) == "").count()
empty_user_id = df_payments.filter(F.trim(F.col("user_id")) == "").count()
print(f"\n[Disabled Anomaly 7 Check] Empty subscription_id: {empty_sub_id:,}, Empty user_id: {empty_user_id:,}")

# ============================================================
# SUMMARY: Blended anomaly rate estimation
# ============================================================
total_users = df_users.count()
total_subs = df_subscriptions.count()
total_pays = df_payments.count()

print(f"\n{'=' * 80}")
print(f"ESTIMATED ANOMALY COUNTS (at ~3.5% blended injection rate):")
print(f"  Users: {total_users:,} rows, ~5 anomaly types -> ~{total_users * 0.035 / 5:,.0f} per type")
print(f"  Subscriptions: {total_subs:,} rows, ~2 anomaly types -> ~{total_subs * 0.035 / 2:,.0f} per type")
print(f"  Payments: {total_pays:,} rows, ~6 anomaly types -> ~{total_pays * 0.035 / 6:,.0f} per type")
print(f"{'=' * 80}")


Reading data from Bronze Layer...
Loaded table users: (10,000 rows)
Loaded table subscription: (11,504 rows)
Loaded table payments: (6,696 rows)
DIRTY DATA INJECTION PROFILING — POSTGRESQL TABLES

USERS TABLE — Dirty Data Injection Profiling

[User Anomaly 1] Null email: 61
[User Anomaly 5] Empty/whitespace email: 71

[User Anomaly 2] Invalid country='XXX': 57

Full country distribution (including dirty):
+-------+-----+
|country|count|
+-------+-----+
|MAR    |1489 |
|KWT    |1438 |
|UAE    |1422 |
|JOR    |1421 |
|KSA    |1398 |
|QAT    |1390 |
|EGY    |1385 |
|XXX    |57   |
+-------+-----+


[User Anomaly 3] last_login before created_at: 68
+------------------------------------+-------------------+-------------------+
|user_id                             |created_at         |last_login         |
+------------------------------------+-------------------+-------------------+
|01a1dedc-2922-72ba-811d-1c36d17f2b36|2026-01-01 00:13:56|2025-12-23 00:13:56|
|01a1dedc-2b00-74de-993c-182f84

In [1]:

# JupyterLab Silver Validation Cell
# Run after Silver pipelines have persisted tables to MinIO

from pyspark.sql import SparkSession
from scripts.storage.config import get_spark_session, SourceSystem
from scripts.storage.config.io_utils import read_silver_table
from scripts.storage.silver.validation import (
    validate_users_silver,
    validate_subscriptions_silver,
    validate_payments_silver,
    validate_events_silver,
    validate_videos_silver,
)

# 1) Initialize Spark
spark = get_spark_session("silver_validation")

# 2) Read all 5 Silver tables
df_users         = read_silver_table(spark, SourceSystem.POSTGRES, "users")
df_subscriptions = read_silver_table(spark, SourceSystem.POSTGRES, "subscriptions")
df_payments      = read_silver_table(spark, SourceSystem.POSTGRES, "payments")
df_events        = read_silver_table(spark, SourceSystem.KAFKA,    "events")
df_videos        = read_silver_table(spark, SourceSystem.MONGO,    "videos")

# 3) Run validators
results = {
    "users":          validate_users_silver(df_users),
    "subscriptions":  validate_subscriptions_silver(df_subscriptions),
    "payments":       validate_payments_silver(df_payments),
    "events":         validate_events_silver(df_events),
    "videos":         validate_videos_silver(df_videos),
}

# 4) Print Pass/Fail summary
print("=" * 70)
print("SILVER VALIDATION SUMMARY")
print("=" * 70)

for table, suite in results.items():
    status = "PASS" if suite.all_passed else "FAIL"
    print(f"\n[{status}] {table.upper()} — {suite.passed} passed, {suite.failed} failed")
    if not suite.all_passed:
        for err in suite.errors:
            print(f"  ✗ {err}")

print("\n" + "=" * 70)

# Optional: raise on any failure (uncomment to halt notebook on failure)
# for table, suite in results.items():
#     suite.raise_if_failed(f"silver:{table}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 04:29:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/16 04:29:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

SILVER VALIDATION SUMMARY

[FAIL] USERS — 9 passed, 1 failed
  ✗ Column 'country' has 57 values outside allowed set

[FAIL] SUBSCRIPTIONS — 7 passed, 1 failed
  ✗ Column 'plan_type' has 212 values outside allowed set

[FAIL] PAYMENTS — 9 passed, 2 failed
  ✗ Column 'payment_status' has 60 values outside allowed set
  ✗ Column 'billing_reason' has 44 values outside allowed set

[FAIL] EVENTS — 16 passed, 2 failed
  ✗ Column 'interaction_type' has 60 values outside allowed set
  ✗ Column 'device_type' has 39 values outside allowed set

[FAIL] VIDEOS — 23 passed, 1 failed
  ✗ Column 'content_type' has 35 values outside allowed set

